In [1]:
# # Belief: what the robot knows (notebook)
#
# The hallway of the Note: 10 cells of 1 m in a loop, doors at cells 1, 3 and 7. The door sensor reads "door" with
# probability 0.6 in front of a door and 0.2 at a wall. The command "move 2" moves the robot 1, 2 or 3 cells with
# probability 0.1, 0.8, 0.1. We check every number of the Note and try other readings ourselves.

In [2]:
import itertools

import numpy as np
import plotly.graph_objects as go

N = 10
doors = np.array([0, 1, 0, 1, 0, 0, 0, 1, 0, 0])          # 1 = door in front of the cell
P_DOOR = {1: 0.6, 0: 0.2}                                 # p(reads door | door cell), p(reads door | wall cell)


def p_z_given_x(z, x):
    """Measurement probability p(z | x): z = 1 'door', z = 0 'wall'."""
    p = P_DOOR[doors[x]]
    return p if z == 1 else 1 - p


# State transition probability for the command "move 2" as a 10 x 10 matrix: T[i, j] = p(x_t = j | x_t-1 = i)
T = np.zeros((N, N))
for i in range(N):
    for step, p in ((1, 0.1), (2, 0.8), (3, 0.1)):
        T[i, (i + step) % N] += p
STAY = np.eye(N)                                          # the command "stay": the robot does not move
print("row of cell 1:", T[1])
print("every row adds to 1:", np.allclose(T.sum(axis=1), 1))

row of cell 1: [0.  0.  0.1 0.8 0.1 0.  0.  0.  0.  0. ]
every row adds to 1: True


## 1. The set-valued information state (Note, Section 3)

With a sensor that never errs, the reading "door" means "one of cells 1, 3, 7". A move takes every possible cell to
every cell it can reach (union); a reading keeps only the cells that agree with it (intersection).

In [3]:
H_door = {x for x in range(N) if doors[x] == 1}
S1 = set(range(N)) & H_door
S2_moved = set().union(*({(x + s) % N for s in (1, 2, 3)} for x in S1))
S2 = S2_moved & H_door
print("after 'door':", sorted(S1))
print("after the move:", sorted(S2_moved))
print("after 'door' again:", sorted(S2))
# With the real sensor, 'door' is possible in every cell (0.2 at a wall), so the reading removes nothing
H_door_noisy = {x for x in range(N) if p_z_given_x(1, x) > 0}
print("cells where a noisy 'door' is possible:", sorted(H_door_noisy))

after 'door': [1, 3, 7]
after the move: [0, 2, 3, 4, 5, 6, 8, 9]
after 'door' again: [3]
cells where a noisy 'door' is possible: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


## 2. The belief after each event (Note, Section 4)

In [4]:
b0 = np.full(N, 0.1)


def read(bel_bar, z):
    u = np.array([p_z_given_x(z, x) for x in range(N)]) * bel_bar
    return u / u.sum()


b1 = read(b0 @ STAY, 1)
bb2 = b1 @ T
b2 = read(bb2, 1)
for name, b in (("bel(x0)", b0), ("bel(x1)", b1), ("predicted bel(x2)", bb2), ("bel(x2)", b2)):
    print(f"{name:18s}", np.round(b, 4), " sum =", round(b.sum(), 6))
print("door shares 3/16 and wall shares 1/16:", 3 / 16, 1 / 16)

bel(x0)            [0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1]  sum = 1.0
bel(x1)            [0.0625 0.1875 0.0625 0.1875 0.0625 0.0625 0.0625 0.1875 0.0625 0.0625]  sum = 1.0
predicted bel(x2)  [0.075  0.0625 0.075  0.1625 0.0875 0.1625 0.075  0.0625 0.075  0.1625]  sum = 1.0
bel(x2)            [0.0476 0.119  0.0476 0.3095 0.0556 0.1032 0.0476 0.119  0.0476 0.1032]  sum = 1.0
door shares 3/16 and wall shares 1/16: 0.1875 0.0625


## 3. The probability of one whole story (Note, Section 6.4)

A story is a list of true cells x0, x1, x2 together with the readings. Its probability is the product of one
factor per arrow of the dynamic Bayes network.

In [5]:
def story_prob(x0, x1, x2, z1=1, z2=1):
    return 0.1 * STAY[x0, x1] * p_z_given_x(z1, x1) * T[x1, x2] * p_z_given_x(z2, x2)


print("start 1 -> 1 -> 3:", round(story_prob(1, 1, 3), 6))
print("start 7 -> 7 -> 9:", round(story_prob(7, 7, 9), 6))
# Adding the stories that end in each cell gives the belief of Section 4
ends = np.zeros(N)
for x0, x1, x2 in itertools.product(range(N), repeat=3):
    ends[x2] += story_prob(x0, x1, x2)
print("stories ending in cell 3:", round(ends[3], 6), " all stories:", round(ends.sum(), 6))
print("their ratio:", round(ends[3] / ends.sum(), 4), " = bel(x2 = 3):", round(b2[3], 4))
assert np.allclose(ends / ends.sum(), b2)

start 1 -> 1 -> 3: 0.0288
start 7 -> 7 -> 9: 0.0096
stories ending in cell 3: 0.0312  all stories: 0.1008
their ratio: 0.3095  = bel(x2 = 3): 0.3095


## 4. Try it: a different reading sequence

Change `readings` (1 = door, 0 = wall) and see how the belief changes after each move of 2 cells.

In [6]:
readings = [1, 1, 0, 0, 1]
bel = read(b0, readings[0])
beliefs = [bel]
for z in readings[1:]:
    bel = read(bel @ T, z)
    beliefs.append(bel)
fig = go.Figure([go.Bar(x=list(range(N)), y=b, name=f"after reading {k + 1}") for k, b in enumerate(beliefs)])
fig.update_layout(barmode="group", template="simple_white", xaxis_title="cell", yaxis_title="belief",
                  title="Belief after each reading (a move of 2 cells between readings)")
fig.show()